<a href="https://colab.research.google.com/github/joanby/tensorflow2/blob/master/Colab%205%20-%20Construir%20una%20Red%20Neuronal%20Recurrente%20en%20TensorFlow%202.0.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

## Paso 1: Instalar las dependencias y la configuración del notebook en GPU

In [57]:
#!pip install tensorflow-gpu==2.0.0.alpha0
%tensorflow_version 2.x

Colab only includes TensorFlow 2.x; %tensorflow_version has no effect.


## Paso 2: Importar las librerías necesarias

In [58]:
import tensorflow as tf
import pandas
import re
import nltk
import numpy

from sklearn.model_selection import train_test_split
from nltk.corpus import stopwords

In [59]:
nltk.download('stopwords')

tf.__version__

[nltk_data] Downloading package stopwords to /root/nltk_data...
[nltk_data]   Package stopwords is already up-to-date!


'2.20.0'

## Paso 3: Pre procesado de datos


### Carga csv ejercicio y preparación

In [60]:
from google.colab import drive
drive.mount('/content/drive')

dataframe = pandas.read_csv('/content/drive/MyDrive/Colab Notebooks/googleplaystore_user_reviews-220414-114436.csv')

dataframe = dataframe.dropna()
dataframe.head()

dataframe = dataframe[['Translated_Review','Sentiment']]
dataframe.head()

def limpia(sen):
    sentence = re.sub(r"\s+[a-zA-Z]\s+", ' ', sen)
    sentence = re.sub(r'\s+', ' ', sentence)
    sentence = re.sub('[^a-zA-Z]', ' ', sentence)
    sentence = sentence.lower()

    words = sentence.split()
    filtered_words = [word for word in words if word not in stopwords.words('english')]

    return ' '.join(filtered_words)

dataframe['Translated_Review'] = dataframe['Translated_Review'].apply(lambda sen: limpia(sen))

def determine_class(label):
  if label == 'Positive':
    return 0
  elif label == 'Neutral':
    return 1
  elif label == 'Negative':
    return 2

REMOVE_NEUTRAL = False
MERGE_NEGATIVE_NEUTRAL = False

if REMOVE_NEUTRAL:
  indexNames = dataframe[dataframe['Sentiment'] == 'Neutral'].index
  dataframe.drop(indexNames , inplace=True)

  y = dataframe['Sentiment'].apply(lambda x: 1 if x == 'Positive' else 0).to_numpy()
else:
  if MERGE_NEGATIVE_NEUTRAL:
    y = dataframe['Sentiment'].apply(lambda x: 1 if x == 'Positive' else 0).to_numpy()
  else:
    y = dataframe['Sentiment'].apply(lambda x: determine_class(x)).to_numpy()

X = dataframe['Translated_Review']
y = y.astype(numpy.uint8)

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


### Cortar secuencias de texto de la misma longitud

In [61]:
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.20, random_state=42)


### Configurar parámetros de la capa de Embedding

In [62]:
from tensorflow.keras.preprocessing.sequence import pad_sequences
from tensorflow.keras.preprocessing.text import Tokenizer

vocab_size = 30000

tokenizer = Tokenizer(num_words=vocab_size)
tokenizer.fit_on_texts(X_train)

X_train = tokenizer.texts_to_sequences(X_train)
X_test = tokenizer.texts_to_sequences(X_test)

X_train = pad_sequences(X_train, padding="post", maxlen=200)
X_test = pad_sequences(X_test, padding="post", maxlen=200)

## Paso 4: Construir la Red Neuronal Recurrente

### Definir el modelo

In [63]:
model = tf.keras.Sequential()



### Añadir la capa de embedding

In [64]:

embed_size = 128

model.add(tf.keras.layers.Embedding(vocab_size, embed_size, input_shape=(X_train.shape[1],)))

/usr/local/lib/python3.12/dist-packages/keras/src/layers/core/embedding.py:103: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(**kwargs)


### Añadir la capa de LSTM

- unidades: 128
- función de activación: tanh

In [65]:
model.add(tf.keras.layers.LSTM(units=128, activation='tanh'))

### Añadir la capa totalmente conectada de salida

- unidades: 1
- función de activación: sigmoid

In [66]:
model.add(tf.keras.layers.Dense(units=1, activation='sigmoid'))

### Compilar el modelo

In [67]:
model.compile(optimizer='rmsprop', loss='binary_crossentropy', metrics=['accuracy'])

In [68]:
model.summary()

Model: "sequential"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ embedding (Embedding)           │ (None, 200, 128)       │     3,840,000 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ lstm (LSTM)                     │ (None, 128)            │       131,584 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense (Dense)                   │ (None, 1)              │           129 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 3,971,713 (15.15 MB)

 Trainable params: 3,971,713 (15.15 MB)

 Non-trainable params: 0 (0.00 B)

### Entrenar el modelo

In [69]:
model.fit(X_train, y_train, epochs=3, batch_size=128)

Epoch 1/3
234/234 ━━━━━━━━━━━━━━━━━━━━ 9s 18ms/step - accuracy: 0.1387 - loss: 0.6818
Epoch 2/3
234/234 ━━━━━━━━━━━━━━━━━━━━ 4s 17ms/step - accuracy: 0.1372 - loss: 0.6811
Epoch 3/3
234/234 ━━━━━━━━━━━━━━━━━━━━ 4s 18ms/step - accuracy: 0.1372 - loss: 0.6815


### Evaluar el modelo

In [70]:
test_loss, test_acurracy = model.evaluate(X_test, y_test)

234/234 ━━━━━━━━━━━━━━━━━━━━ 2s 8ms/step - accuracy: 0.1401 - loss: 0.6803


In [71]:
print("Test accuracy: {}".format(test_acurracy))

Test accuracy: 0.1401282399892807
